In [4]:
pip install pygad

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 2.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import pygad
import io

df = pd.read_csv("Financial_Budgeting_Dataset.csv")

df_agg = df.groupby("Department").agg({
    "Actual_Revenue": "mean",
    "Budget_Allocated": "sum"
}).reset_index()

df_agg["return_rate"] = df_agg["Actual_Revenue"] / df_agg["Budget_Allocated"]

departments = df_agg["Department"].tolist()
returns = df_agg["return_rate"].values
limits = df_agg["Budget_Allocated"].values
n = len(departments)

TOTAL_BUDGET = 500000
MIN_SHARE = 0.01 * TOTAL_BUDGET
MAX_SHARE = 0.40 * TOTAL_BUDGET

def fitness_func(ga_instance, solution, solution_idx):
    total_spent = np.sum(solution)
    expected_return = np.sum(solution * returns)

    if total_spent > TOTAL_BUDGET:
        return expected_return - (total_spent - TOTAL_BUDGET) * 10
    return expected_return

gene_space = []
for i in range(n):
    upper_bound = min(limits[i], MAX_SHARE)
    gene_space.append({'low': MIN_SHARE, 'high': upper_bound})

ga_instance = pygad.GA(
    num_generations=500,
    num_parents_mating=10,
    fitness_func=fitness_func,
    sol_per_pop=100,
    num_genes=n,
    gene_space=gene_space,
    parent_selection_type="rank",
    crossover_type="uniform",
    mutation_type="random",
    mutation_num_genes=1,
    keep_elitism=3
)

ga_instance.run()

best_sol, best_fit, _ = ga_instance.best_solution()

print("--- Final Heuristic Allocation Report ---")
print(f"Status:         {'SUCCESS' if np.sum(best_sol) <= TOTAL_BUDGET else 'OVER BUDGET'}")
print(f"Total Budget:   {TOTAL_BUDGET:,.2f}")
print(f"Used Budget:    {np.sum(best_sol):,.2f}")
print(f"Fitness Score:  {best_fit:,.4f}")
print("-" * 40)

for i, dept in enumerate(departments):
    print(f"{dept:12}: {best_sol[i]:>10,.2f} (ROI: {returns[i]:.4f})")

--- Final Heuristic Allocation Report ---
Status:         SUCCESS
Total Budget:   500,000.00
Used Budget:    460,580.14
Fitness Score:  634.8431
----------------------------------------
Finance     :  11,963.00 (ROI: 0.0014)
HR          :  99,707.01 (ROI: 0.0013)
IT          :  85,812.75 (ROI: 0.0014)
Logistics   :  36,380.17 (ROI: 0.0014)
Marketing   :   8,854.87 (ROI: 0.0014)
Operations  :  82,968.64 (ROI: 0.0014)
R&D         :  57,404.12 (ROI: 0.0014)
Sales       :  77,489.58 (ROI: 0.0014)


In [ ]:
import pandas as pd
from pulp import *

# Load dataset
df = pd.read_csv("Financial_Budgeting_Dataset.csv")

# Aggregate by department
df = df.groupby("Department").agg({
    "Actual_Revenue": "mean",
    "Budget_Allocated": "sum"
}).reset_index()

# Create return rate
df["return_rate"] = df["Actual_Revenue"] / df["Budget_Allocated"]

departments = df["Department"].tolist()
returns = df["return_rate"].tolist()
limits = df["Budget_Allocated"].tolist()

TOTAL_BUDGET = 500000

MIN_SHARE = 0.01 * TOTAL_BUDGET
MAX_SHARE = 0.40 * TOTAL_BUDGET

# Model
model = LpProblem("Budget_Allocation", LpMaximize)

x = LpVariable.dicts("Budget", departments, lowBound=0)

# Objective
model += lpSum(x[d] * returns[i] for i, d in enumerate(departments))

# Budget constraint
model += lpSum(x[d] for d in departments) <= TOTAL_BUDGET

# Department constraints
for i, d in enumerate(departments):
    model += x[d] <= min(limits[i], MAX_SHARE)
    model += x[d] >= MIN_SHARE

model.solve()

print("Status:", LpStatus[model.status])
for d in departments:
    print(d, x[d].value())